In [1]:
import requests, json
import truststore; truststore.inject_into_ssl()     # แก้ SSLError จาก certificate chain ของ server

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
HEADERS = {"User-Agent": "pm25-explorer/0.1 (student project)"}

r = requests.get(URL, headers=HEADERS, timeout=(10, 60))
print(r.status_code, r.headers.get("Content-Type"), len(r.content), "bytes")
data = r.json()

# ถ้าดึงไม่ได้ ใช้ไฟล์ตัวอย่างของรายวิชา
# data = json.load(open("../data/sample/air4thai_sample.json", encoding="utf-8"))

200 application/json; charset=UTF-8 140970 bytes


In [3]:
print(list(data.keys()))
st = data["stations"]
print(len(st), "สถานี")
print(json.dumps(st[0], ensure_ascii=False, indent=2)[:1500])   # ดูสถานีแรกแบบอ่านง่าย

['stations']
181 สถานี
{
  "stationID": "119t",
  "nameTH": "สวนสาธารณะธารา",
  "nameEN": "Thara Public Park",
  "areaTH": "ต.ปากน้ำ อ.เมือง, กระบี่",
  "areaEN": "Pak Nam Subdistrict, Mueang District, Krabi",
  "stationType": "GROUND",
  "lat": "8.0506237",
  "long": "98.9180489",
  "forecast": [],
  "AQILast": {
    "date": "2026-11-28",
    "time": "14:00",
    "PM25": {
      "color_id": "1",
      "aqi": "13",
      "value": "7.9"
    },
    "PM10": {
      "color_id": "0",
      "aqi": "-1",
      "value": "-1"
    },
    "O3": {
      "color_id": "1",
      "aqi": "22",
      "value": "31"
    },
    "CO": {
      "color_id": "1",
      "aqi": "3",
      "value": "0.60"
    },
    "NO2": {
      "color_id": "1",
      "aqi": "0",
      "value": "1"
    },
    "SO2": {
      "color_id": "1",
      "aqi": "1",
      "value": "2"
    },
    "AQI": {
      "color_id": "1",
      "aqi": "22",
      "param": "O3"
    }
  }
}


| คอลัมน์ในตารางของเรา | path ของ key ใน JSON | ชนิดใน JSON | ตัวอย่างค่า |
|---|---|---|---|
| `station_id` | `stations[].stationID` | string | `119t` |
| `name_th` / `name_en` | `stations[].nameTH` / `stations[].nameEN` | string / string | `สวนสาธารณะธารา` / `Thara Public Park` |
| `province` | ไม่มี key จังหวัดแยก; จังหวัดอยู่ในข้อความ `stations[].areaTH` (หรือ `areaEN`) | string | `ต.ปากน้ำ อ.เมือง, กระบี่` |
| `lat` / `lon` | `stations[].lat` / `stations[].long` | string / string | `8.0506237` / `98.9180489` |
| `station_type` | `stations[].stationType` | string | `GROUND` |
| วันที่ + เวลาของค่าล่าสุด | `stations[].AQILast.date` / `stations[].AQILast.time` | string / string | `2026-11-28` / `14:00` |
| `pm25` (ความเข้มข้น ไม่ใช่ AQI) | `stations[].AQILast.PM25.value` | string | `7.9` (ส่วน AQI แยกอยู่ที่ `...PM25.aqi` เช่น `13`) |
| `pm10`, `o3`, `co`, `no2`, `so2` | `stations[].AQILast.PM10.value`, `.O3.value`, `.CO.value`, `.NO2.value`, `.SO2.value` | string ทั้งหมด | `-1`, `31`, `0.60`, `1`, `2` |

**ค่าหายและความสม่ำเสมอ:** พบ `"-1"` (เป็น string) ใช้แทนค่าที่ไม่มี เช่น `PM10.value` และ `PM10.aqi`; อย่าตีความเป็นค่าศูนย์หรือค่าที่วัดได้ ให้แปลงเป็น missing (`NaN`/`NA`) ตอนเตรียมข้อมูล โดยตรวจ marker อื่นจากข้อมูลจริงด้วย ในไฟล์ตัวอย่างมี 178 สถานีและ key ระดับสถานี, `AQILast`, วันเวลา และ key ของมลพิษทั้งหกครบทุกสถานี ส่วนผลที่แสดงในโน้ตบุ๊กนี้พิมพ์รายละเอียดเพียงสถานีแรก จึงควรตรวจทุกสถานีจาก response จริงก่อนสรุปความสม่ำเสมอของ API รอบนั้น


In [4]:
from collections import Counter
Counter(tuple(sorted(s.keys())) for s in st).most_common(3)   # รูปแบบ key ที่พบบ่อย

[(('AQILast',
   'areaEN',
   'areaTH',
   'forecast',
   'lat',
   'long',
   'nameEN',
   'nameTH',
   'stationID',
   'stationType'),
  181)]

In [ ]:
from urllib.robotparser import RobotFileParser

ROBOTS_URL = "http://air4thai.pcd.go.th/robots.txt"
robots_response = requests.get(ROBOTS_URL, headers=HEADERS, timeout=(10, 60))
print("robots.txt:", robots_response.status_code, robots_response.url)

if robots_response.status_code == 200:
    rp = RobotFileParser()
    rp.set_url(robots_response.url)
    rp.parse(robots_response.text.splitlines())
    print("can_fetch:", rp.can_fetch(HEADERS["User-Agent"], URL))
else:
    print(f"Cannot determine robots rules (HTTP {robots_response.status_code}); do not infer permission.")

**ผลตรวจ robots.txt (2026-11-28):** URL เปลี่ยนเส้นทางไป HTTPS แล้วตอบ `404 Not Found` จึงไม่มี directives ให้อ่านที่ path นี้ และไม่ควรตีความ `can_fetch=True` จาก parser ที่ไม่มี rules ว่าเป็นการอนุญาตอย่างชัดแจ้ง ทั้งนี้ `robots.txt` เป็นแนวทางสำหรับ crawler ไม่ใช่ใบอนุญาตด้านลิขสิทธิ์หรือเงื่อนไขการนำข้อมูลไปเผยแพร่ต่อ


In [5]:
from urllib.robotparser import RobotFileParser
rp = RobotFileParser("http://air4thai.pcd.go.th/robots.txt")
rp.read()
rp.can_fetch(HEADERS["User-Agent"], URL)

True